# 结构化输出小结

目前观察到一个比较关键的现象：**模型的结构化输出能力不能只看“能不能输出 JSON”，还要区分强提示词和弱提示词。**

在强提示词下，例如明确要求：

```text
生成一个关于《星际穿越》的电影信息，包含导演、演员、评分
```

Flash 模型在 Pydantic、TypedDict、JSON Schema、dataclass 四种结构化方式下都可能通过。这说明它并不是完全没有结构化遵循能力；当字段要求已经被提示词明确写出来时，它有能力按结构输出。

但在弱提示词下，例如只说：

```text
请介绍电影《盗梦空间》
```

模型需要自己理解任务，并根据 schema 主动补齐 `title/year/director/cast/rating` 等必填字段。这个时候 Flash 更容易只返回局部字段，例如只返回标题；甚至 Pro 在 `method="json_schema"` 下也可能失败。这说明部分供应商的 JSON Schema 支持更像是“强结构提示”，不一定等同于 OpenAI strict JSON Schema 那种强约束解码。

因此测试结构化输出时要分两层看：

1. **格式遵循能力**：模型是否能输出合法 JSON / dict / Pydantic 可解析对象。
2. **语义补全能力**：模型是否能在弱提示词下，根据 schema 主动补齐所有必填字段。

强提示词测的是“模型在明确字段要求下能不能做到”；弱提示词更能暴露“schema 本身能不能兜住模型输出”。

整体经验：

- **Pydantic**：最推荐，Python 侧校验最明确，失败会直接抛错。
- **TypedDict**：轻量，适合快速声明 dict 结构，但运行时需要自己补校验。
- **JSON Schema**：跨模型兼容性通常较强，但不等于一定严格校验。
- **dataclass**：写法简洁，但生态和校验能力不如 Pydantic，作为了解即可。


In [ ]:
### Pydantic：三模型 + 强/弱提示词验证

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from pydantic import BaseModel, Field, ValidationError
from rich import print as rich_print
from typing import List
import os

load_dotenv(override=True)


class Actor(BaseModel):
    """演员信息"""
    name: str = Field(description="演员姓名")
    role: str = Field(description="饰演的角色")


class Movie(BaseModel):
    """电影信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="上映年份")
    director: str = Field(description="导演")
    cast: List[Actor] = Field(description="演员列表")
    rating: float = Field(description="评分，满分十分")


def validate_movie_result(result) -> tuple[dict | None, list[str]]:
    if isinstance(result, Movie):
        return result.model_dump(), []
    if isinstance(result, dict):
        try:
            return Movie.model_validate(result).model_dump(), []
        except ValidationError as e:
            return None, [str(e)]
    return None, [f"返回值既不是 Movie 实例，也不是 dict，而是 {type(result).__name__}"]


models = {
    "deepseek-v4-flash": init_chat_model(
        model="deepseek-v4-flash",
        extra_body={"thinking": {"type": "disabled"}},
    ),
    "deepseek-v4-pro": init_chat_model(
        model="deepseek-v4-pro",
        extra_body={"thinking": {"type": "disabled"}},
    ),
    "gpt-5.4-mini": init_chat_model(
        model="gpt-5.4-mini",
        model_provider="openai",
        api_key=os.getenv("OPENROUTER_API_KEY"),
        base_url=os.getenv("OPENROUTER_BASE_URL"),
    ),
}


strong_prompt = "生成一个关于《星际穿越》的电影信息，包含导演、演员、评分"
weak_prompt = "请介绍电影《盗梦空间》"

for prompt_name, prompt in {"强提示词": strong_prompt, "弱提示词": weak_prompt}.items():
    print(f"\n######## {prompt_name} ########")
    for model_name, llm in models.items():
        print(f"\n===== {model_name} =====")
        try:
            structured_model = llm.with_structured_output(Movie)
            response = structured_model.invoke(prompt)
            data, errors = validate_movie_result(response)

            if errors:
                print("[FAIL] Pydantic 结构校验失败")
                rich_print(errors)
            else:
                print("[PASS] Pydantic 结构校验通过")
                rich_print(data)

            rich_print(response)

        except Exception as e:
            print("[FAIL] 调用失败")
            print(type(e).__name__, e)


In [ ]:
### TypedDict：三模型 + 强/弱提示词验证

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rich_print
from typing import List
from typing_extensions import TypedDict, Annotated
import os

load_dotenv(override=True)


class ActorDict(TypedDict):
    """演员信息"""
    name: Annotated[str, ..., "演员姓名"]
    role: Annotated[str, ..., "饰演的角色"]


class MovieDict(TypedDict):
    """电影信息"""
    title: Annotated[str, ..., "电影标题"]
    year: Annotated[int, ..., "上映年份"]
    director: Annotated[str, ..., "导演"]
    cast: Annotated[List[ActorDict], ..., "演员列表"]
    rating: Annotated[float, ..., "评分，满分十分"]


def validate_movie_result(result: dict) -> list[str]:
    errors = []
    if not isinstance(result, dict):
        return [f"返回值不是 dict，而是 {type(result).__name__}"]

    required_fields = ["title", "year", "director", "cast", "rating"]
    for field in required_fields:
        if field not in result:
            errors.append(f"缺少必填字段: {field}")

    if "title" in result and not isinstance(result["title"], str):
        errors.append("title 应该是 string")
    if "year" in result and (not isinstance(result["year"], int) or isinstance(result["year"], bool)):
        errors.append("year 应该是 integer")
    if "director" in result and not isinstance(result["director"], str):
        errors.append("director 应该是 string")
    if "rating" in result and (not isinstance(result["rating"], (int, float)) or isinstance(result["rating"], bool)):
        errors.append("rating 应该是 number")

    cast = result.get("cast")
    if "cast" in result and not isinstance(cast, list):
        errors.append("cast 应该是 array")
    elif isinstance(cast, list):
        for index, actor in enumerate(cast):
            if not isinstance(actor, dict):
                errors.append(f"cast[{index}] 应该是 object")
                continue
            if not isinstance(actor.get("name"), str):
                errors.append(f"cast[{index}].name 应该是 string")
            if not isinstance(actor.get("role"), str):
                errors.append(f"cast[{index}].role 应该是 string")
    return errors


models = {
    "deepseek-v4-flash": init_chat_model(
        model="deepseek-v4-flash",
        extra_body={"thinking": {"type": "disabled"}},
    ),
    "deepseek-v4-pro": init_chat_model(
        model="deepseek-v4-pro",
        extra_body={"thinking": {"type": "disabled"}},
    ),
    "gpt-5.4-mini": init_chat_model(
        model="gpt-5.4-mini",
        model_provider="openai",
        api_key=os.getenv("OPENROUTER_API_KEY"),
        base_url=os.getenv("OPENROUTER_BASE_URL"),
    ),
}


strong_prompt = "生成一个关于《星际穿越》的电影信息，包含导演、演员、评分"
weak_prompt = "请介绍电影《盗梦空间》"

for prompt_name, prompt in {"强提示词": strong_prompt, "弱提示词": weak_prompt}.items():
    print(f"\n######## {prompt_name} ########")
    for model_name, llm in models.items():
        print(f"\n===== {model_name} =====")
        try:
            structured_model = llm.with_structured_output(MovieDict,include_raw=True) # include_raw打印模型输出的原始信息
            response = structured_model.invoke(prompt) 
            errors = validate_movie_result(response)

            if errors:
                print("[FAIL] TypedDict 结构校验失败")
                rich_print(errors)
            else:
                print("[PASS] TypedDict 结构校验通过")

            rich_print(response)

        except Exception as e:
            print("[FAIL] 调用失败")
            print(type(e).__name__, e)


In [ ]:
### JSON Schema：三模型 + 强/弱提示词验证

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rich_print
import os

load_dotenv(override=True)


movie_schema = {
    "title": "MovieInfo",
    "description": "包含电影标题、上映年份、导演、演员和评分的电影对象",
    "type": "object",
    "properties": {
        "title": {"type": "string", "description": "电影标题"},
        "year": {"type": "integer", "description": "上映年份"},
        "director": {"type": "string", "description": "导演"},
        "cast": {
            "type": "array",
            "description": "演员列表",
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "description": "演员姓名"},
                    "role": {"type": "string", "description": "演员角色"},
                },
                "required": ["name", "role"],
            },
        },
        "rating": {"type": "number", "description": "评分，满分十分"},
    },
    "required": ["title", "year", "director", "cast", "rating"],
}


def validate_movie_result(result: dict) -> list[str]:
    errors = []
    if not isinstance(result, dict):
        return [f"返回值不是 dict，而是 {type(result).__name__}"]

    required_fields = ["title", "year", "director", "cast", "rating"]
    for field in required_fields:
        if field not in result:
            errors.append(f"缺少必填字段: {field}")

    if "title" in result and not isinstance(result["title"], str):
        errors.append("title 应该是 string")
    if "year" in result and (not isinstance(result["year"], int) or isinstance(result["year"], bool)):
        errors.append("year 应该是 integer")
    if "director" in result and not isinstance(result["director"], str):
        errors.append("director 应该是 string")
    if "rating" in result and (not isinstance(result["rating"], (int, float)) or isinstance(result["rating"], bool)):
        errors.append("rating 应该是 number")

    cast = result.get("cast")
    if "cast" in result and not isinstance(cast, list):
        errors.append("cast 应该是 array")
    elif isinstance(cast, list):
        for index, actor in enumerate(cast):
            if not isinstance(actor, dict):
                errors.append(f"cast[{index}] 应该是 object")
                continue
            if not isinstance(actor.get("name"), str):
                errors.append(f"cast[{index}].name 应该是 string")
            if not isinstance(actor.get("role"), str):
                errors.append(f"cast[{index}].role 应该是 string")
    return errors


models = {
    "deepseek-v4-flash": init_chat_model(
        model="deepseek-v4-flash",
        extra_body={"thinking": {"type": "disabled"}},
    ),
    "deepseek-v4-pro": init_chat_model(
        model="deepseek-v4-pro",
        extra_body={"thinking": {"type": "disabled"}},
    ),
    "gpt-5.4-mini": init_chat_model(
        model="gpt-5.4-mini",
        model_provider="openai",
        api_key=os.getenv("OPENROUTER_API_KEY"),
        base_url=os.getenv("OPENROUTER_BASE_URL"),
    ),
}


strong_prompt = "生成一个关于《星际穿越》的电影信息，包含导演、演员、评分"
weak_prompt = "请介绍电影《盗梦空间》"

for prompt_name, prompt in {"强提示词": strong_prompt, "弱提示词": weak_prompt}.items():
    print(f"\n######## {prompt_name} ########")
    for model_name, llm in models.items():
        print(f"\n===== {model_name} =====")
        try:
            structured_model = llm.with_structured_output(schema=movie_schema, method="json_schema")
            response = structured_model.invoke(prompt)
            errors = validate_movie_result(response)

            if errors:
                print("[FAIL] JSON Schema 结构校验失败")
                rich_print(errors)
            else:
                print("[PASS] JSON Schema 结构校验通过")

            rich_print(response)

        except Exception as e:
            print("[FAIL] 调用失败")
            print(type(e).__name__, e)


In [ ]:
### dataclass：三模型 + 强/弱提示词验证

from dataclasses import asdict, dataclass, field, is_dataclass
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rich_print
from typing import List
import os

load_dotenv(override=True)


@dataclass
class Actor:
    """演员信息"""
    name: str = field(metadata={"description": "演员姓名"})
    role: str = field(metadata={"description": "饰演的角色"})


@dataclass
class Movie:
    """电影信息"""
    title: str = field(metadata={"description": "电影标题"})
    year: int = field(metadata={"description": "上映年份"})
    director: str = field(metadata={"description": "导演"})
    cast: List[Actor] = field(metadata={"description": "演员列表"})
    rating: float = field(metadata={"description": "评分，满分十分"})


def normalize_movie_result(result):
    if is_dataclass(result) and not isinstance(result, type):
        return asdict(result), []
    if isinstance(result, dict):
        return result, []
    return None, [f"返回值既不是 dataclass 实例，也不是 dict，而是 {type(result).__name__}"]


def validate_movie_result(result) -> list[str]:
    data, errors = normalize_movie_result(result)
    if errors:
        return errors

    required_fields = ["title", "year", "director", "cast", "rating"]
    for field_name in required_fields:
        if field_name not in data:
            errors.append(f"缺少必填字段: {field_name}")

    if "title" in data and not isinstance(data["title"], str):
        errors.append("title 应该是 string")
    if "year" in data and (not isinstance(data["year"], int) or isinstance(data["year"], bool)):
        errors.append("year 应该是 integer")
    if "director" in data and not isinstance(data["director"], str):
        errors.append("director 应该是 string")
    if "rating" in data and (not isinstance(data["rating"], (int, float)) or isinstance(data["rating"], bool)):
        errors.append("rating 应该是 number")

    cast = data.get("cast")
    if "cast" in data and not isinstance(cast, list):
        errors.append("cast 应该是 array")
    elif isinstance(cast, list):
        for index, actor in enumerate(cast):
            if not isinstance(actor, dict):
                errors.append(f"cast[{index}] 应该是 object")
                continue
            if not isinstance(actor.get("name"), str):
                errors.append(f"cast[{index}].name 应该是 string")
            if not isinstance(actor.get("role"), str):
                errors.append(f"cast[{index}].role 应该是 string")
    return errors


models = {
    "deepseek-v4-flash": init_chat_model(
        model="deepseek-v4-flash",
        extra_body={"thinking": {"type": "disabled"}},
    ),
    "deepseek-v4-pro": init_chat_model(
        model="deepseek-v4-pro",
        extra_body={"thinking": {"type": "disabled"}},
    ),
    "gpt-5.4-mini": init_chat_model(
        model="gpt-5.4-mini",
        model_provider="openai",
        api_key=os.getenv("OPENROUTER_API_KEY"),
        base_url=os.getenv("OPENROUTER_BASE_URL"),
    ),
}


strong_prompt = "生成一个关于《星际穿越》的电影信息，包含导演、演员、评分"
weak_prompt = "请介绍电影《盗梦空间》"

for prompt_name, prompt in {"强提示词": strong_prompt, "弱提示词": weak_prompt}.items():
    print(f"\n######## {prompt_name} ########")
    for model_name, llm in models.items():
        print(f"\n===== {model_name} =====")
        try:
            structured_model = llm.with_structured_output(Movie)
            response = structured_model.invoke(prompt)
            errors = validate_movie_result(response)

            if errors:
                print("[FAIL] dataclass 结构校验失败")
                rich_print(errors)
            else:
                print("[PASS] dataclass 结构校验通过")

            rich_print(response)

        except Exception as e:
            print("[FAIL] 调用失败")
            print(type(e).__name__, e)
